In [14]:
import pandas as pd
from collections import Counter
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('omw-1.4')

# 下载必要的NLTK数据
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

class SimpleTextAnalyzer:
    def __init__(self, file_path):
        """初始化分析器"""
        self.df = pd.read_excel(file_path)
        self.prepare_data()
        
    def prepare_data(self):
        """数据预处理"""
        print("数据列名:", self.df.columns.tolist())
        print("数据形状:", self.df.shape)
        
        # 寻找suggestion列
        suggestion_col = None
        for col in self.df.columns:
            if 'suggestion' in col.lower():
                suggestion_col = col
                break
        
        if not suggestion_col:
            print("错误：未找到'suggestion'列！")
            print("可用的列名:", self.df.columns.tolist())
            return
        
        self.suggestion_col = suggestion_col
        print(f"找到建议列: {suggestion_col}")
        
        # 清理空值
        self.df = self.df.dropna(subset=[suggestion_col])
        print(f"清理空值后剩余数据: {len(self.df)} 条")
        
        print("前几行suggestion数据:")
        for i, text in enumerate(self.df[suggestion_col].head().tolist(), 1):
            print(f"{i}. {str(text)[:100]}{'...' if len(str(text)) > 100 else ''}")
        
    def extract_top_words(self, text_list, top_n=10):
        """提取文本中最常见的词汇"""
        # 合并所有文本
        all_text = ' '.join([str(text) for text in text_list if pd.notna(text)])
        
        if not all_text.strip():
            return []
        
        # 预处理文本
        text = all_text.lower()
        # 移除标点符号和数字，保留字母
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        
        # 分词
        tokens = word_tokenize(text)
        
        # 移除停用词
        stop_words = set(stopwords.words('english'))
        # 添加常见无意义词
        custom_stopwords = {
            'would', 'could', 'should', 'also', 'really', 'maybe', 'think', 'like', 
            'get', 'go', 'one', 'two', 'three', 'use', 'way', 'time', 'good', 'new', 
            'first', 'make', 'work', 'see', 'come', 'back', 'much', 'well', 'little', 
            'need', 'want', 'take', 'give', 'know', 'say', 'said', 'thing', 'things', 
            'people', 'person', 'year', 'years', 'day', 'days', 'week', 'weeks', 
            'month', 'months', 'feel', 'felt', 'feeling', 'lot', 'bit', 'kind', 
            'sort', 'help', 'try', 'trying', 'something', 'anything', 'everything'
        }
        stop_words.update(custom_stopwords)
        
        # 过滤词汇：移除停用词，保留长度>2的词
        filtered_tokens = [
            token for token in tokens 
            if token not in stop_words and len(token) > 2 and token.isalpha()
        ]
        
        # 词形还原
        lemmatizer = WordNetLemmatizer()
        lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
        
        # 计算词频
        word_freq = Counter(lemmatized_tokens)
        
        # 返回最常见的词
        return word_freq.most_common(top_n)
    
    def analyze_by_role(self):
        """按角色分析最常提到的内容"""
        print("\n" + "="*50)
        print("按父母角色分析suggestion列中最常提到的10个词")
        print("="*50)
        
        if not hasattr(self, 'suggestion_col'):
            print("错误：未找到suggestion列")
            return
        
        # 寻找角色列
        role_col = None
        for col in self.df.columns:
            if any(keyword in col.lower() for keyword in ['role', 'parent', 'father', 'mother', 'dad', 'mom']):
                role_col = col
                break
        
        if not role_col:
            print("未找到角色列，跳过角色分析")
            return
        
        roles = self.df[role_col].unique()
        print(f"发现的角色: {roles}")
        
        for role in roles:
            role_data = self.df[self.df[role_col] == role]
            role_suggestions = role_data[self.suggestion_col].tolist()
            
            print(f"\n【{role}】在suggestion列中最常提到的10个词:")
            print(f"数据条数: {len(role_data)}")
            
            top_words = self.extract_top_words(role_suggestions, 10)
            
            if top_words:
                for i, (word, count) in enumerate(top_words, 1):
                    print(f"{i:2d}. {word:<15} (提到 {count} 次)")
            else:
                print("    没有找到有效词汇")
    
    def analyze_by_group(self):
        """按组别分析最常提到的内容"""
        print("\n" + "="*50)
        print("按组别分析suggestion列中最常提到的10个词")
        print("="*50)
        
        if not hasattr(self, 'suggestion_col'):
            print("错误：未找到suggestion列")
            return
        
        # 寻找组别列
        group_col = None
        for col in self.df.columns:
            if any(keyword in col.lower() for keyword in ['group', 'condition', 'treatment']):
                group_col = col
                break
        
        if not group_col:
            print("未找到组别列，跳过组别分析")
            return
        
        groups = self.df[group_col].unique()
        print(f"发现的组别: {groups}")
        
        for group in groups:
            group_data = self.df[self.df[group_col] == group]
            group_suggestions = group_data[self.suggestion_col].tolist()
            
            print(f"\n【{group}组】在suggestion列中最常提到的10个词:")
            print(f"数据条数: {len(group_data)}")
            
            top_words = self.extract_top_words(group_suggestions, 10)
            
            if top_words:
                for i, (word, count) in enumerate(top_words, 1):
                    print(f"{i:2d}. {word:<15} (提到 {count} 次)")
            else:
                print("    没有找到有效词汇")
    
    def get_sample_texts(self):
        """显示一些样本文本以便检查"""
        print("\n" + "="*50)
        print("suggestion列样本文本预览")
        print("="*50)
        
        if not hasattr(self, 'suggestion_col'):
            print("错误：未找到suggestion列")
            return
        
        print("前5条suggestion内容:")
        for i, text in enumerate(self.df[self.suggestion_col].head().tolist(), 1):
            print(f"{i}. {str(text)[:100]}{'...' if len(str(text)) > 100 else ''}")

def main():
    """主函数"""
    print("开始分析suggestion列...")
    
    # 初始化分析器
    analyzer = SimpleTextAnalyzer('/Users/isabellawu/Downloads/1m_suggestions en.xlsx')
    
    # 检查是否成功找到suggestion列
    if not hasattr(analyzer, 'suggestion_col'):
        print("程序终止：未找到suggestion列")
        return
    
    # 显示样本文本
    analyzer.get_sample_texts()
    
    # 按角色分析suggestion列
    analyzer.analyze_by_role()
    
    # 按组别分析suggestion列
    analyzer.analyze_by_group()
    
    print("\nsuggestion列分析完成！")

if __name__ == "__main__":
    main()

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/isabellawu/nltk_data...
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/isabellawu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


开始分析suggestion列...
数据列名: ['suggestion', 'mother', 'group', 'Unnamed: 3']
数据形状: (966, 4)
找到建议列: suggestion
清理空值后剩余数据: 962 条
前几行suggestion数据:
1. Very professional
2. The baby is taken care of by a confinement nanny for the first month, and the parents are inexperien...
3. I think the time the father spends with the baby during the period when the confinement nanny lives ...
4. More open resources can be provided to help new parents understand and learn
5. Thank you for the training and knowledge you provide to the participants.

suggestion列样本文本预览
前5条suggestion内容:
1. Very professional
2. The baby is taken care of by a confinement nanny for the first month, and the parents are inexperien...
3. I think the time the father spends with the baby during the period when the confinement nanny lives ...
4. More open resources can be provided to help new parents understand and learn
5. Thank you for the training and knowledge you provide to the participants.

按父母角色分析suggestion列中最常提到的10个词
发现的角色: ['M

In [2]:
pip install jieba

     |████████████████████████████████| 19.2 MB 923 kB/s eta 0:00:01
  Created wheel for jieba: filename=jieba-0.42.1-py3-none-any.whl size=19314478 sha256=db1a96d9cc2720a34792dfb7d7b5e6f490b9491e4c307adc1a4b238038ca0394
  Stored in directory: /Users/isabellawu/Library/Caches/pip/wheels/7d/74/cf/08c94db4b784e2c1ef675a600b7b5b281fd25240dcb954ee7e
Successfully built jieba
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install WordCloud

     |████████████████████████████████| 172 kB 553 kB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install textstat

     |████████████████████████████████| 175 kB 390 kB/s eta 0:00:01
     |████████████████████████████████| 939 kB 238 kB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 340 kB/s eta 0:00:01
  Attempting uninstall: zipp
    Found existing installation: zipp 3.7.0
    Uninstalling zipp-3.7.0:
      Successfully uninstalled zipp-3.7.0
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib-metadata 4.11.3
    Uninstalling importlib-metadata-4.11.3:
      Successfully uninstalled importlib-metadata-4.11.3
Note: you may need to restart the kernel to use updated packages.
